# AI-Based Predictive Auto-Scaling Using Machine Learning

**Practical:** Predict future application workload and simulate automatic scaling of virtual-machine instances.

This notebook:
1. Generates sample workload data or imports your own CSV.
2. Creates time-series features.
3. Trains a Random Forest regression model.
4. Predicts the next 5-minute workload.
5. Simulates scale-out and scale-in decisions.
6. Visualizes workload, capacity, and scaling activity.
7. Exports the simulation results as a CSV file.

**Important:** This is a simulation. It does not provision or delete real cloud VMs. The instance capacity and scaling thresholds are illustrative and should be calibrated using real application measurements.

## 1. Install and import libraries

Run this cell first.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from math import ceil
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## 2. Generate sample data or upload your own CSV

The sample dataset contains 14 days of observations at 5-minute intervals, with daily patterns, noise, and occasional workload spikes.

To upload your own CSV, set `USE_UPLOAD = True`. Your file must contain:
- `timestamp`
- `requests_per_sec`

The timestamps should be in chronological order. The notebook assumes evenly spaced 5-minute observations.

In [ ]:
USE_UPLOAD = False  # Change to True to upload your own CSV

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    df = pd.read_csv(filename)
else:
    np.random.seed(42)

    periods = 14 * 24 * 12  # 14 days, 5-minute intervals
    timestamps = pd.date_range(
        start="2026-11-01",
        periods=periods,
        freq="5min"
    )

    time = np.arange(periods)

    # Daily pattern: lower and higher demand at different times of day
    daily_pattern = 60 + 35 * np.sin(2 * np.pi * time / (24 * 12))

    # Slower weekly variation
    weekly_pattern = 15 * np.sin(2 * np.pi * time / (7 * 24 * 12))

    # Random variation
    noise = np.random.normal(0, 12, periods)

    requests = 70 + daily_pattern + weekly_pattern + noise

    # Add occasional workload spikes
    spike_indices = np.random.choice(periods, size=35, replace=False)
    requests[spike_indices] += np.random.uniform(70, 150, size=35)

    requests = np.maximum(requests, 5)

    df = pd.DataFrame({
        "timestamp": timestamps,
        "requests_per_sec": requests
    })

# Validate and clean the dataset
required_columns = {"timestamp", "requests_per_sec"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

df = df[["timestamp", "requests_per_sec"]].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["requests_per_sec"] = pd.to_numeric(df["requests_per_sec"], errors="coerce")
df = df.dropna().sort_values("timestamp").reset_index(drop=True)

if len(df) < 50:
    raise ValueError("Please provide at least 50 valid observations.")

if (df["requests_per_sec"] < 0).any():
    raise ValueError("requests_per_sec cannot contain negative values.")

print("Dataset loaded successfully.")
print("Rows:", len(df))
display(df.head())

plt.figure(figsize=(14, 4))
plt.plot(df["timestamp"], df["requests_per_sec"])
plt.title("Historical Application Workload")
plt.xlabel("Time")
plt.ylabel("Requests per second")
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Feature engineering

The model uses previous workload values, rolling averages, and time-of-day/day-of-week features to predict the workload in the **next observation**.

For the generated dataset, each observation is 5 minutes apart. If you upload your own data, keep the interval consistent.

In [ ]:
def create_features(data):
    data = data.copy()

    # Lag features: earlier workload values
    for lag in [1, 2, 3, 6, 12]:
        data[f"lag_{lag}"] = data["requests_per_sec"].shift(lag)

    # Rolling averages, based only on current and past observations
    for window in [3, 6, 12]:
        data[f"rolling_mean_{window}"] = (
            data["requests_per_sec"].rolling(window=window).mean()
        )

    # Cyclical time features
    hour = data["timestamp"].dt.hour + data["timestamp"].dt.minute / 60
    day = data["timestamp"].dt.dayofweek

    data["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    data["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    data["day_sin"] = np.sin(2 * np.pi * day / 7)
    data["day_cos"] = np.cos(2 * np.pi * day / 7)

    # Target is workload at the next observation
    data["target"] = data["requests_per_sec"].shift(-1)

    return data

feature_data = create_features(df).dropna().reset_index(drop=True)

feature_columns = [
    col for col in feature_data.columns
    if col not in ["timestamp", "target"]
]

X = feature_data[feature_columns]
y = feature_data["target"]

print("Features created:", feature_columns)
print("Usable samples:", len(X))

## 4. Train and evaluate the Machine Learning model

A chronological 80:20 split is used. Earlier observations train the model and later observations test it. This is more appropriate for time-series data than randomly shuffling observations.

In [ ]:
split_index = int(len(X) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MODEL TRAINING COMPLETED")
print("------------------------")
print(f"MAE:  {mae:.2f} requests/sec")
print(f"RMSE: {rmse:.2f} requests/sec")
print(f"R²:   {r2:.3f}")

test_timestamps = feature_data["timestamp"].iloc[split_index:].reset_index(drop=True)

plt.figure(figsize=(14, 5))
plt.plot(test_timestamps, y_test.reset_index(drop=True), label="Actual workload")
plt.plot(test_timestamps, predictions, label="Predicted workload", linestyle="--")
plt.title("Actual vs Predicted Workload")
plt.xlabel("Time")
plt.ylabel("Requests per second")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Configure and simulate predictive auto-scaling

Assumptions:
- One instance can handle 100 requests/second.
- Target utilization is 65%.
- Minimum instances: 1.
- Maximum instances: 10.
- Scale-in threshold: 35%.
- Cooldown: 2 observations.

The controller calculates the number of instances needed for the predicted workload. It scales out immediately when more capacity is required and scales in by at most one instance when utilization is low. A cooldown prevents repeated changes too quickly.

In [ ]:
# Auto-scaling configuration (illustrative values)
CAPACITY_PER_INSTANCE = 100
TARGET_UTILIZATION = 0.65
SCALE_DOWN_THRESHOLD = 0.35

MIN_INSTANCES = 1
MAX_INSTANCES = 10
INITIAL_INSTANCES = 2

COOLDOWN_PERIOD = 2

current_instances = INITIAL_INSTANCES
cooldown = 0
scaling_results = []

for i, predicted_load in enumerate(predictions):
    predicted_load = max(0.0, float(predicted_load))
    actual_load = max(0.0, float(y_test.iloc[i]))

    predicted_utilization = predicted_load / (
        current_instances * CAPACITY_PER_INSTANCE
    )

    desired_instances = ceil(
        predicted_load / (CAPACITY_PER_INSTANCE * TARGET_UTILIZATION)
    )
    desired_instances = max(
        MIN_INSTANCES,
        min(MAX_INSTANCES, desired_instances)
    )

    action = "NO CHANGE"

    if cooldown > 0:
        cooldown -= 1
    else:
        if desired_instances > current_instances:
            current_instances = desired_instances
            action = "SCALE OUT"
            cooldown = COOLDOWN_PERIOD

        elif (
            predicted_utilization < SCALE_DOWN_THRESHOLD
            and desired_instances < current_instances
        ):
            current_instances = max(
                desired_instances,
                current_instances - 1
            )
            action = "SCALE IN"
            cooldown = COOLDOWN_PERIOD

    actual_utilization = (
        actual_load / (current_instances * CAPACITY_PER_INSTANCE)
    )

    scaling_results.append({
        "timestamp": test_timestamps.iloc[i],
        "actual_load": actual_load,
        "predicted_load": predicted_load,
        "instances": current_instances,
        "predicted_utilization_pct": predicted_utilization * 100,
        "actual_utilization_pct": actual_utilization * 100,
        "action": action
    })

results = pd.DataFrame(scaling_results)

print("AI AUTO-SCALING SIMULATION COMPLETED")
display(results.head(20))

## 6. Visualize workload and scaling decisions

The available capacity line is the number of instances multiplied by the assumed capacity per instance.

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

ax1.plot(
    results["timestamp"],
    results["actual_load"],
    label="Actual workload",
    linewidth=1.4
)
ax1.plot(
    results["timestamp"],
    results["predicted_load"],
    label="Predicted workload",
    linestyle="--",
    linewidth=1.4
)
ax1.plot(
    results["timestamp"],
    results["instances"] * CAPACITY_PER_INSTANCE,
    label="Available capacity",
    linestyle=":",
    linewidth=2
)

ax1.set_xlabel("Time")
ax1.set_ylabel("Requests per second")
ax1.set_title("AI-Based Predictive Auto-Scaling")
ax1.grid(True)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.step(
    results["timestamp"],
    results["instances"],
    where="post",
    label="Running instances"
)
ax2.set_ylabel("Number of instances")
ax2.set_ylim(0, MAX_INSTANCES + 1)

plt.tight_layout()
plt.show()

## 7. Scaling summary and decisions

In [ ]:
scale_out_count = (results["action"] == "SCALE OUT").sum()
scale_in_count = (results["action"] == "SCALE IN").sum()

print("========== SCALING SUMMARY ==========")
print("Scale-out events:", scale_out_count)
print("Scale-in events:", scale_in_count)
print("Minimum instances used:", results["instances"].min())
print("Maximum instances used:", results["instances"].max())
print("Average instances:", round(results["instances"].mean(), 2))
print(
    "Average actual utilization:",
    round(results["actual_utilization_pct"].mean(), 2),
    "%"
)

print("\nScaling decisions:")
decision_rows = results[results["action"] != "NO CHANGE"]
display(decision_rows[[
    "timestamp", "predicted_load", "instances", "action"
]].head(30))

if decision_rows.empty:
    print("No scaling changes were required for this test period.")

## 8. Export results to CSV

This creates a CSV file containing the actual workload, predicted workload, instance count, utilization, and scaling action.

In [ ]:
output_file = "AI_AutoScaling_Results.csv"
results.to_csv(output_file, index=False)

print(f"Results saved to: {output_file}")

try:
    from google.colab import files
    files.download(output_file)
except ImportError:
    print("Not running in Google Colab. The CSV is saved in the current working directory.")

## 9. Result

The Random Forest regression model was trained on historical workload data to predict the next observation's request rate. Based on the predicted workload, the auto-scaling simulator calculated the required number of instances and generated scale-out, scale-in, or no-change decisions.

## 10. Conclusion

Thus, AI-based predictive auto-scaling was implemented and simulated using Python and Machine Learning in Google Colab.

## Important limitation

This notebook demonstrates the prediction and decision logic only. It does **not** launch or terminate real virtual machines. A production implementation would require integration with a cloud control plane such as Kubernetes or OpenStack, authentication, monitoring, safety limits, and testing against real workload data.